# RFB lakehouse — silver (SCD2), gold e linhagem

Abra pelo VS Code (Remote-SSH `lambdaserver`), kernel **Python (lambda-data)**. Execute depois que a DAG `rfb_pipeline`
tiver rodado ao menos um mês.

| Camada | Banco (metastore) | O que tem |
|---|---|---|
| bronze | `rfb_bronze` | o mês como a Receita publicou (strings), partição `_ref_month` |
| silver | `rfb_silver` | `*_hist` = histórico SCD2 (uma linha por **versão**); listas de códigos (SCD1) |
| gold | `rfb_gold` | `estabelecimento_atual`, `socio_atual`, `fato_mudanca_cadastral`, `agg_estabelecimentos_mensal`, `dim_*` |
| controle | `ctrl` | downloads e watermarks por camada/mês |
| auditoria | Postgres `audit` (+ Delta `audit.quarantine`) | execuções, tasks, checks, logs, estatísticas das tabelas |

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
from pipelines.commons.lake import duckdb_lake, list_views

REF = 202609
con = duckdb_lake()
[v for v in list_views(con) if v.startswith(("rfb_silver", "rfb_gold"))]

## 1. SCD2: versões, versão atual e "como era em um mês"
Cada linha de `estabelecimento_hist` é uma **versão**: `valid_from`/`valid_to` (YYYYMM), `is_current`,
`change_type` (NEW/CHANGED) e `closed_reason` (CHANGED/ABSENT).

In [ ]:
con.sql("""
    SELECT count(*) AS versoes, count(*) FILTER (WHERE is_current) AS atuais,
           count(DISTINCT cnpj) AS cnpjs, min(valid_from) AS primeiro_mes, max(valid_from) AS ultimo_mes
    FROM rfb_silver.estabelecimento_hist
""").df()

In [ ]:
# histórico completo de um CNPJ (troque pelo que quiser investigar)
CNPJ = "11222333000181"
con.sql(f"""
    SELECT valid_from, valid_to, is_current, change_type, closed_reason, situacao_cadastral, nome_fantasia,
           cnae_fiscal_principal, logradouro, municipio, uf
    FROM rfb_silver.estabelecimento_hist WHERE cnpj = '{CNPJ}' ORDER BY valid_from
""").df()

In [ ]:
# "as of": a versão válida em um mês (a mesma regra que a gold usa para construir qualquer mês)
M = REF
con.sql(f"""
    SELECT situacao_cadastral, count(*) AS estabelecimentos
    FROM rfb_silver.estabelecimento_hist
    WHERE valid_from <= {M} AND (valid_to IS NULL OR valid_to >= {M})
    GROUP BY 1 ORDER BY 2 DESC
""").df()

## 2. Gold: consumo

In [ ]:
con.sql(f"""
    SELECT cnpj, razao_social, nome_fantasia, situacao_cadastral_desc, cnae_principal_desc, porte_empresa_desc,
           opcao_simples, opcao_mei, municipio_nome, uf, idade_anos
    FROM rfb_gold.estabelecimento_atual WHERE uf = 'SP' LIMIT 10
""").df()

In [ ]:
# o que mudou no mês, por entidade e atributo (vazio no primeiro mês: é a linha de base)
con.sql(f"""
    SELECT entidade, tipo_evento, coalesce(atributo, '-') AS atributo, count(*) AS eventos
    FROM rfb_gold.fato_mudanca_cadastral WHERE ref_month = {REF}
    GROUP BY ALL ORDER BY eventos DESC LIMIT 30
""").df()

In [ ]:
con.sql(f"""
    SELECT uf, sum(qtd_estabelecimentos) AS total, sum(qtd_abertos_no_mes) AS abertos, sum(qtd_baixados_no_mes) AS baixados
    FROM rfb_gold.agg_estabelecimentos_mensal WHERE _ref_month = {REF}
    GROUP BY uf ORDER BY total DESC
""").df()

## 3. Spark: histórico de versões, CDF e propriedades
O Delta guarda cada commit: `DESCRIBE HISTORY` mostra operação e métricas; o **Change Data Feed** devolve as linhas
alteradas entre versões; a propriedade `lambda.scd2.last_month` diz qual mês a tabela aplicou por último.

In [ ]:
from pipelines.commons.spark import build_spark

spark = build_spark("notebook-lakehouse", master="local[4]", conf={"spark.ui.showConsoleProgress": "false"})
spark.sql("DESCRIBE HISTORY rfb_silver.estabelecimento_hist").select(
    "version", "timestamp", "operation", "operationMetrics.numTargetRowsInserted",
    "operationMetrics.numTargetRowsMatchedUpdated", "operationMetrics.numTargetRowsNotMatchedBySourceUpdated"
).show(truncate=False)

In [ ]:
spark.sql("SHOW TBLPROPERTIES rfb_silver.estabelecimento_hist").where("key LIKE 'lambda.%' OR key LIKE 'delta.%'").show(truncate=False)

In [ ]:
# CDF: o que o último MERGE fez, linha a linha (insert / update_preimage / update_postimage)
last = spark.sql("DESCRIBE HISTORY rfb_silver.estabelecimento_hist LIMIT 1").first()["version"]
(spark.sql(f"SELECT _change_type, count(*) AS linhas FROM table_changes('rfb_silver.estabelecimento_hist', {last}, {last}) GROUP BY 1")
 .show())

In [ ]:
spark.stop()

## 4. Linhagem: qual script escreveu o quê, em qual execução
`audit.task_run` liga a task do Airflow ao script (task_id) e ao Spark app; `audit.delta_table_stats` liga a task à
**versão** Delta que ela criou. Com o `run_id` você sabe exatamente de onde veio cada versão de cada tabela.

In [ ]:
import psycopg

with psycopg.connect("dbname=lambda host=/var/run/postgresql") as pg:
    run_id = pg.execute("SELECT run_id FROM audit.pipeline_run WHERE pipeline='rfb' ORDER BY started_at DESC LIMIT 1").fetchone()[0]
    rows = pg.execute("""
        SELECT t.airflow_task_id, t.task_id AS script_task, t.layer, t.spark_app_id, s.table_name, s.operation,
               s.delta_version, s.rows_written, round(t.duration_s) AS seconds
        FROM audit.task_run t LEFT JOIN audit.delta_table_stats s ON s.run_id = t.run_id AND s.task_id = t.task_id
        WHERE t.run_id = %s ORDER BY t.started_at, s.captured_at""", (run_id,)).fetchall()
print(run_id)
import pandas as pd
pd.DataFrame(rows, columns=["airflow_task", "script_task", "layer", "spark_app", "table", "operation", "version",
                            "rows_written", "seconds"])